# Notebook 05: Silver Enrichment
## Purpose: Enrich sensor features with machine metadata + maintenance history
## Input:  silver_sensor_features + bronze_machine_metadata + bronze_maintenance_logs
## Output: workspace.predictive_maintenance.silver_machine_enriched
## Key Steps: Clean metadata, JOIN on unit_id, aggregate maintenance history


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window

# Load tables
sensor_df = spark.table("workspace.predictive_maintenance.silver_sensor_features")
metadata_df = spark.table("workspace.predictive_maintenance.bronze_machine_metadata")
maint_df = spark.table("workspace.predictive_maintenance.bronze_maintenance_logs")

print(f" silver_sensor_features:    {sensor_df.count():,} rows | {len(sensor_df.columns)} cols")
print(f" bronze_machine_metadata:   {metadata_df.count():,} rows | {len(metadata_df.columns)} cols")
print(f" bronze_maintenance_logs:   {maint_df.count():,} rows | {len(maint_df.columns)} cols")

# Preview metadata columns
print("\n=== METADATA COLUMNS ===")
metadata_df.printSchema()

print("\n=== MAINTENANCE COLUMNS ===")
maint_df.printSchema()

In [0]:
# See what's in metadata before joining
print("=== MACHINE METADATA SAMPLE ===")
display(metadata_df.limit(5))

print("\n=== UNIQUE MACHINES IN METADATA ===")
print(f"Unique machineIDs: {metadata_df.select('machineID').distinct().count()}")

print("\n=== UNIQUE MACHINES IN SENSOR DATA ===")
print(f"Unique unit_ids: {sensor_df.select('unit_id').distinct().count()}")

In [0]:
# Rename machineID to unit_id for joining
# Azure dataset uses machineID, NASA uses unit_id

# Get column names
print("Metadata columns:", metadata_df.columns)

# Aggregate metadata per machine — take averages since 
# Azure telemetry has multiple rows per machine
metadata_clean = metadata_df \
    .groupBy("machineID") \
    .agg(
        F.avg("volt").alias("avg_volt"),
        F.avg("rotate").alias("avg_rotate"),
        F.avg("pressure").alias("avg_pressure"),
        F.avg("vibration").alias("avg_vibration")
    ) \
    .withColumnRenamed("machineID", "unit_id")

# Handle nulls
metadata_clean = metadata_clean.fillna(0)

print(f" Metadata cleaned: {metadata_clean.count()} unique machines")
display(metadata_clean.limit(5))

In [0]:
print("=== MAINTENANCE LOGS SAMPLE ===")
display(maint_df.limit(5))

# Aggregate maintenance history per machine
maint_agg = maint_df \
    .groupBy("machineID") \
    .agg(
        F.count("*").alias("maintenance_count"),
        F.countDistinct("comp").alias("unique_components_replaced")
    ) \
    .withColumnRenamed("machineID", "unit_id")

# Fill machines with no maintenance history
maint_agg = maint_agg.fillna(0)

print(f"\n Maintenance aggregated: {maint_agg.count()} machines")
print(f"   Features created:")
print(f"   - maintenance_count: total times serviced")
print(f"   - unique_components_replaced: variety of maintenance done")
display(maint_agg.limit(5))

In [0]:
# Step 1: Join sensor features with metadata
enriched_df = sensor_df.join(
    metadata_clean,
    on="unit_id",
    how="left"
)

print(f" After metadata join: {enriched_df.count():,} rows | {len(enriched_df.columns)} cols")

# Step 2: Join with maintenance aggregations
enriched_df = enriched_df.join(
    maint_agg,
    on="unit_id",
    how="left"
)

print(f" After maintenance join: {enriched_df.count():,} rows | {len(enriched_df.columns)} cols")

# Fill nulls from left joins (machines with no match)
enriched_df = enriched_df.fillna(0)

print(f"\n New context columns added:")
print(f"   From metadata: avg_volt, avg_rotate, avg_pressure, avg_vibration")
print(f"   From maintenance: maintenance_count, unique_components_replaced")

In [0]:
print("=== JOIN QUALITY CHECK ===")

total = enriched_df.count()
with_maint = enriched_df.filter(F.col("maintenance_count") > 0).count()
no_maint = enriched_df.filter(F.col("maintenance_count") == 0).count()

print(f"Total rows:            {total:,}")
print(f"Rows with maintenance: {with_maint:,} ({with_maint/total*100:.1f}%)")
print(f"Rows without:          {no_maint:,} ({no_maint/total*100:.1f}%)")

print("\n=== SAMPLE ENRICHED ROW ===")
enriched_df.select(
    "unit_id", "cycle", "RUL", "fail_30",
    "avg_volt", "avg_vibration",
    "maintenance_count", "sensor_deviation_score"
).show(5)

In [0]:
# Write as Delta table
enriched_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.predictive_maintenance.silver_machine_enriched")

# OPTIMIZE
spark.sql("""
    OPTIMIZE workspace.predictive_maintenance.silver_machine_enriched
    ZORDER BY (unit_id, cycle)
""")

count = spark.table("workspace.predictive_maintenance.silver_machine_enriched").count()
cols  = len(spark.table("workspace.predictive_maintenance.silver_machine_enriched").columns)

print(f" silver_machine_enriched written + optimized!")
print(f"   Rows:    {count:,}")
print(f"   Columns: {cols}")

In [0]:
# Show version history — demonstrates Delta Lake schema evolution
print("=== DESCRIBE HISTORY — silver_sensor_features ===")
spark.sql("""
    DESCRIBE HISTORY 
    workspace.predictive_maintenance.silver_sensor_features
""").select("version", "timestamp", "operation") \
  .show(10, truncate=False)

print("\n Multiple versions visible — Delta Lake tracks every change")
print(" This is your schema evolution demo for the video")

In [0]:
print("=" * 55)
print("NOTEBOOK 05 COMPLETE — SILVER ENRICHMENT")
print("=" * 55)

tables = [
    "silver_sensor_cleaned",
    "silver_sensor_features", 
    "silver_machine_enriched"
]

for t in tables:
    count = spark.table(f"workspace.predictive_maintenance.{t}").count()
    cols  = len(spark.table(f"workspace.predictive_maintenance.{t}").columns)
    print(f" {t}")
    print(f"   {count:,} rows | {cols} columns")

print("=" * 55)
print("Silver Layer: 3 tables complete")
print("Next: Notebook 06 — Feature Store")
print("=" * 55)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

sample = spark.table("workspace.predictive_maintenance.silver_machine_enriched") \
    .sample(0.1, seed=42) \
    .toPandas()

# Drop non-numeric columns before correlation
non_numeric = ['source_dataset']
sample_numeric = sample.drop(columns=non_numeric, errors='ignore')

# Correlation vs RUL
correlations = sample_numeric.corr()['RUL'] \
    .drop('RUL') \
    .abs() \
    .sort_values(ascending=False)

print("=== TOP 15 FEATURES CORRELATED WITH RUL ===")
print(correlations.head(15))

fig, ax = plt.subplots(figsize=(10, 6))
correlations.head(15).plot(kind='bar', ax=ax, color='#1565C0')
ax.set_title("Top 15 Features — Correlation with RUL",
             fontweight='bold', fontsize=13)
ax.set_xlabel("Feature")
ax.set_ylabel("Absolute Correlation")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print("✅ Feature importance preview done!")

# 📋 Notebook 05: Silver Enrichment

## Business Context
Raw sensor features alone don't tell the full story.
A machine that has NEVER been serviced behaving unusually
is far more concerning than one serviced last week.
This notebook adds that operational context.

## What This Notebook Does
1. Aggregates Azure telemetry into per-machine averages
2. Aggregates maintenance history per machine
3. LEFT JOINs both onto silver_sensor_features
4. Writes silver_machine_enriched — the final ML-ready Silver table

## Why LEFT JOIN?
We use LEFT JOIN to keep ALL sensor records.
Machines with no metadata match get 0-filled context columns.
No rows are lost — data completeness is preserved.

## Output Table: silver_machine_enriched
| Column Group | Count | Source |
|---|---|---|
| Sensor features | 51 | Notebook 04 |
| Operating settings | 3 | NASA CMAPSS |
| Telemetry averages | 4 | Azure metadata |
| Maintenance history | 2 | Azure maint logs |
| Target variables | 2 | Engineered |